This notebook processes the data:
- load data
- check dataset
- unwrap lists & dict in columns
- remove duplicate
- handle missing values
- fix data types
- handle outliers
- normalize data
- clean categorical data

In [2]:
import json
from pathlib import Path
import pandas as pd

# Load Data

In [3]:
data_folder=Path('./data')

def read_json(file_path):
    with open(file_path, 'r') as f: 
        try: 
            return json.load(f) 
        except json.JSONDecodeError:
            f.seek(0)
            return[json.loads(line) for line in f if line.strip()]

In [4]:
portfolio_data = read_json(data_folder/'portfolio.json')
profile_data = read_json(data_folder/'profile.json')
transcript_data = read_json(data_folder/'transcript.json')

### Transform to DataFrame

In [5]:
print(type(portfolio_data))

<class 'list'>


In [6]:
portfolio_data = pd.DataFrame(portfolio_data)
profile_data = pd.DataFrame(profile_data)
transcript_data = pd.DataFrame(transcript_data)

# Pre-processing

## portfolio data

In [7]:
portfolio_data

,reward,channels,difficulty,duration,offer_type,id
0,10,"[email, mobile, social]",10,7.0,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"[web, email, mobile, social]",10,5.0,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"[web, email, mobile]",0,4.0,informational,3f207df678b143eea3cee63160fa8bed
3,5,"[web, email, mobile]",5,7.0,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"[web, email]",20,10.0,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7
5,3,"[web, email, mobile, social]",7,7.0,discount,2298d6c36e964ae4a3e7e9706d1fb8c2
6,2,"[web, email, mobile, social]",10,10.0,discount,fafdcd668e3743c1bb461111dcafc2a4
7,0,"[email, mobile, social]",0,3.0,informational,5a8bc65990b245e5a138643cd4eb9837
8,5,"[web, email, mobile, social]",5,5.0,bogo,f19421c1d4aa40978ebb69ca19b0e20d
9,2,"[web, email, mobile]",10,7.0,discount,2906b810c7d4411798c6938adc9daaa5


In [8]:
portfolio_data.describe()

,reward,difficulty,duration
count,10.000000,10.000000,10.000000
mean,4.200000,7.700000,6.500000
std,3.583915,5.831905,2.321398
min,0.000000,0.000000,3.000000
25%,2.000000,5.000000,5.000000
50%,4.000000,8.500000,7.000000
75%,5.000000,10.000000,7.000000
max,10.000000,20.000000,10.000000


In [9]:
portfolio_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   reward      10 non-null     int64  
 1   channels    10 non-null     object 
 2   difficulty  10 non-null     int64  
 3   duration    10 non-null     float64
 4   offer_type  10 non-null     object 
 5   id          10 non-null     object 
dtypes: float64(1), int64(2), object(3)
memory usage: 612.0+ bytes


In [10]:
portfolio_data.isnull().sum()

reward        0
channels      0
difficulty    0
duration      0
offer_type    0
id            0
dtype: int64

In [11]:
print('unique values in reward column:', portfolio_data['reward'].unique())
print('unique values in difficulty column:', portfolio_data['difficulty'].unique())
print('unique values in offer_type column:', portfolio_data['offer_type'].unique())
print('dataset shape:', portfolio_data.shape)

unique values in reward column: [10  0  5  3  2]
unique values in difficulty column: [10  0  5 20  7]
unique values in offer_type column: ['bogo' 'informational' 'discount']
dataset shape: (10, 6)


### Transform channels column to 4 columns

In [12]:
# unwrap list
portfolio_data['web']=portfolio_data['channels'].apply(lambda x: 1 if 'web' in x else 0)
portfolio_data['mobile']=portfolio_data['channels'].apply(lambda x: 1 if 'mobile' in x else 0)
portfolio_data['social']=portfolio_data['channels'].apply(lambda x: 1 if 'social' in x else 0)
portfolio_data['email']=portfolio_data['channels'].apply(lambda x: 1 if 'email' in x else 0)    

In [13]:
portfolio_data = portfolio_data.drop(columns=['channels'])

In [14]:
portfolio_data

,reward,difficulty,duration,offer_type,id,web,mobile,social,email
0,10,10,7.0,bogo,ae264e3637204a6fb9bb56bc8210ddfd,0,1,1,1
1,10,10,5.0,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1
2,0,0,4.0,informational,3f207df678b143eea3cee63160fa8bed,1,1,0,1
3,5,5,7.0,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,0,1
4,5,20,10.0,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,0,0,1
5,3,7,7.0,discount,2298d6c36e964ae4a3e7e9706d1fb8c2,1,1,1,1
6,2,10,10.0,discount,fafdcd668e3743c1bb461111dcafc2a4,1,1,1,1
7,0,0,3.0,informational,5a8bc65990b245e5a138643cd4eb9837,0,1,1,1
8,5,5,5.0,bogo,f19421c1d4aa40978ebb69ca19b0e20d,1,1,1,1
9,2,10,7.0,discount,2906b810c7d4411798c6938adc9daaa5,1,1,0,1


In [15]:
portfolio_data.drop_duplicates()

,reward,difficulty,duration,offer_type,id,web,mobile,social,email
0,10,10,7.0,bogo,ae264e3637204a6fb9bb56bc8210ddfd,0,1,1,1
1,10,10,5.0,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1
2,0,0,4.0,informational,3f207df678b143eea3cee63160fa8bed,1,1,0,1
3,5,5,7.0,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,0,1
4,5,20,10.0,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,0,0,1
5,3,7,7.0,discount,2298d6c36e964ae4a3e7e9706d1fb8c2,1,1,1,1
6,2,10,10.0,discount,fafdcd668e3743c1bb461111dcafc2a4,1,1,1,1
7,0,0,3.0,informational,5a8bc65990b245e5a138643cd4eb9837,0,1,1,1
8,5,5,5.0,bogo,f19421c1d4aa40978ebb69ca19b0e20d,1,1,1,1
9,2,10,7.0,discount,2906b810c7d4411798c6938adc9daaa5,1,1,0,1


## profile data

In [16]:
profile_data

,gender,age,id,became_member_on,income
0,None,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,None,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,None,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN
...,...,...,...,...,...
16995,F,45,6d5f3a774f3d4714ab0c092238f3a1d7,20180604,54000.0
16996,M,61,2cb4f97358b841b9a9773a7aa05a9d77,20180713,72000.0
16997,M,49,01d26f638c274aa0b965d24cefe3183f,20170126,73000.0
16998,F,83,9dc1421481194dcd9400aec7c9ae6366,20160307,50000.0


In [17]:
profile_data.describe()

,age,income
count,17000.000000,14825.000000
mean,62.531412,65404.991568
std,26.738580,21598.299410
min,18.000000,30000.000000
25%,45.000000,49000.000000
50%,58.000000,64000.000000
75%,73.000000,80000.000000
max,118.000000,120000.000000


In [18]:
profile_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17000 entries, 0 to 16999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            14825 non-null  object 
 1   age               17000 non-null  int64  
 2   id                17000 non-null  object 
 3   became_member_on  17000 non-null  object 
 4   income            14825 non-null  float64
dtypes: float64(1), int64(1), object(3)
memory usage: 664.2+ KB


In [19]:
profile_data=profile_data.drop_duplicates()

In [20]:
profile_data.isnull().sum()

gender              2175
age                    0
id                     0
became_member_on       0
income              2175
dtype: int64

In [21]:
2175/17000*100

12.794117647058822

In [22]:
profile_data.dropna(inplace=True)

In [23]:
profile_data.isnull().sum()

gender              0
age                 0
id                  0
became_member_on    0
income              0
dtype: int64

## transcript data

In [24]:
transcript_data

,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0
...,...,...,...,...
306529,b3a1272bc9904337b331bf348c3e8c17,transaction,{'amount': 1.59},714
306530,68213b08d99a4ae1b0dcb72aebd9aa35,transaction,{'amount': 9.53},714
306531,a00058cf10334a308c68e7631c529907,transaction,{'amount': 3.61},714
306532,76ddbd6576844afe811f1a3c0fbb5bec,transaction,{'amount': 3.53},714


In [25]:
print('unique value in event column:', transcript_data['event'].unique())

unique value in event column: ['offer received' 'offer viewed' 'transaction' 'offer completed']


In [26]:
transcript_data.describe()

,time
count,306534.000000
mean,366.382940
std,200.326314
min,0.000000
25%,186.000000
50%,408.000000
75%,528.000000
max,714.000000


In [27]:
transcript_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306534 entries, 0 to 306533
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   person  306534 non-null  object
 1   event   306534 non-null  object
 2   value   306534 non-null  object
 3   time    306534 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 9.4+ MB


In [28]:
transcript_data.isnull().sum()

person    0
event     0
value     0
time      0
dtype: int64

### Transform event column to 3 columns

In [29]:
transcript_data['offer_received']=transcript_data['event'].apply(lambda x: 1 if 'offer_received' in x else 0)

In [30]:
transcript_data['offer viewed']=transcript_data['event'].apply(lambda x: 1 if 'offer viewed' in x else 0)
transcript_data['transaction']=transcript_data['event'].apply(lambda x: 1 if 'transaction' in x else 0)
transcript_data['offer completed']=transcript_data['event'].apply(lambda x: 1 if 'offer completed' in x else 0)

In [31]:
transcript_data.drop(columns=['event'], inplace=True)

In [32]:
transcript_data

,person,value,time,offer_received,offer viewed,transaction,offer completed
0,78afa995795e4d85b5d9ceeca43f5fef,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0,0,0,0,0
1,a03223e636434f42ac4c3df47e8bac43,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0,0,0,0,0
2,e2127556f4f64592b11af22de27a7932,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0,0,0,0,0
3,8ec6ce2a7e7949b1bf142def7d0e0586,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0,0,0,0,0
4,68617ca6246f4fbc85e91a2a49552598,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0,0,0,0,0
...,...,...,...,...,...,...,...
306529,b3a1272bc9904337b331bf348c3e8c17,{'amount': 1.59},714,0,0,1,0
306530,68213b08d99a4ae1b0dcb72aebd9aa35,{'amount': 9.53},714,0,0,1,0
306531,a00058cf10334a308c68e7631c529907,{'amount': 3.61},714,0,0,1,0
306532,76ddbd6576844afe811f1a3c0fbb5bec,{'amount': 3.53},714,0,0,1,0


### Transform value column to 3 columns

In [33]:
transcript_data['offer_id'] = transcript_data['value'].apply(lambda x: x.get('offer_id', x.get('offer id', 0)))
transcript_data['amount'] = transcript_data['value'].apply(lambda x: x.get('amount', x.get('amount', 0)))
transcript_data['reward'] = transcript_data['value'].apply(lambda x: x.get('reward', x.get('reward', 0)))

In [34]:
transcript_data.drop(columns='value', inplace=True)

In [35]:
transcript_data

,person,time,offer_received,offer viewed,transaction,offer completed,offer_id,amount,reward
0,78afa995795e4d85b5d9ceeca43f5fef,0,0,0,0,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,0.00,0
1,a03223e636434f42ac4c3df47e8bac43,0,0,0,0,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,0.00,0
2,e2127556f4f64592b11af22de27a7932,0,0,0,0,0,2906b810c7d4411798c6938adc9daaa5,0.00,0
3,8ec6ce2a7e7949b1bf142def7d0e0586,0,0,0,0,0,fafdcd668e3743c1bb461111dcafc2a4,0.00,0
4,68617ca6246f4fbc85e91a2a49552598,0,0,0,0,0,4d5c57ea9a6940dd891ad53e9dbe8da0,0.00,0
...,...,...,...,...,...,...,...,...,...
306529,b3a1272bc9904337b331bf348c3e8c17,714,0,0,1,0,0,1.59,0
306530,68213b08d99a4ae1b0dcb72aebd9aa35,714,0,0,1,0,0,9.53,0
306531,a00058cf10334a308c68e7631c529907,714,0,0,1,0,0,3.61,0
306532,76ddbd6576844afe811f1a3c0fbb5bec,714,0,0,1,0,0,3.53,0


In [36]:
transcript_data.isnull().sum()

person             0
time               0
offer_received     0
offer viewed       0
transaction        0
offer completed    0
offer_id           0
amount             0
reward             0
dtype: int64

In [37]:
transcript_data.drop_duplicates(inplace=True)

# Save Processed Data

In [38]:
portfolio_data.to_csv(data_folder/'portfolio_cleaned.csv', index=False)
profile_data.to_csv(data_folder/'profile_cleaned.csv', index=False)
transcript_data.to_csv(data_folder/'transcript_cleaned.csv', index=False)